# Organoid skeleton construction: step-by-step walkthrough

This notebook walks through building a **biology-aware skeleton graph** for a small batch of intestinal organoids. It follows the same spirit as `tutorial_segmentation.ipynb`: keep intermediate objects visible, make the important parameters easy to edit, and provide quick visual feedback.

The skeleton is not a generic medial axis. It is built from fresh crypt detections and uses straight graph edges only:

- `body -> neck -> tip` for a simple crypt;
- `body -> neck -> bend -> tip` when a bend node is requested;
- `body -> neck -> branch -> daughter neck -> daughter tip` when an initial crypt candidate splits into multiple refined crypts.

The most useful tuning loop is the batch run: change the crypt detection, filtering, refinement, and bend-node parameters, rebuild the skeletons, and inspect each mesh overlay.

In [1]:
# --- Imports ---
from pathlib import Path

import numpy as np

from IPython.display import HTML, display

from organograph.io_utils.dataset_config import load_mesh_dataset_config
from organograph.io_utils.path_parsing import discover_mesh_paths, parse_mesh_path
from organograph.mesh.OrganoidMesh import OrganoidMesh
from organograph.mesh.geodesics import compute_geodesics_dijkstra

from organograph.crypts.filters import filter_crypts_by_hks_percent, filter_crypts_by_size

# During skeleton development, Jupyter may keep an older version of these modules
# in memory. Drop only the skeleton modules so imports below reflect the files on disk.
import importlib
import sys

for _mod in [
    "organograph.skeleton.build",
    "organograph.skeleton.datatypes",
    "organograph.skeleton.geometry",
    "organograph.skeleton.io",
    "organograph.skeleton.primitives",
    "organograph.skeleton.primitive_fitting",
    "organograph.skeleton.primitive_geometry",
    "organograph.skeleton",
]:
    sys.modules.pop(_mod, None)
importlib.invalidate_caches()
from organograph.plotting.skeletons import (
    plot_mesh_with_skeleton,
    plot_mesh_with_skeleton_and_primitives,
)

from organograph.skeleton.build import (
    build_skeleton_from_crypt_detections,
    detect_crypts_for_skeleton,
)
from organograph.skeleton.primitive_fitting import (
    attach_body_primitive,
    attach_branch_primitives,
    attach_crypt_tube_primitives,
    primitive_components_from_crypt_detections,
)




## 1) Dataset paths + organoid selection

Edit this cell first. The tutorial now runs a batch of organoids. Each entry only needs a `timepoint`, `well`, and `organoid_id`; the notebook reads `mesh_config.json` to fill in the zarr, round, and mesh folder names.


In [2]:
# -----------------------
# CONFIG: edit these
# -----------------------

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "notebooks" and (NOTEBOOK_DIR / "notebooks").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
PROJECT_ROOT = NOTEBOOK_DIR.parent

DATASET = "20251201"

MESH_DATA_DIR = PROJECT_ROOT.parent / "NicoleData" / DATASET / "fractal_output"
MESH_CONFIG_PATH = PROJECT_ROOT.parent / "NicoleData" / DATASET / "mesh_config.json"
VOCAB_PATH = PROJECT_ROOT / "sim" / "vocab_with_meta.npz"

# Organoids copied from tutorial_segmentation.ipynb.
ORGANOID_SPECS = [
    {"timepoint": "day4p5", "well": "B03", "organoid_id": "144"},
    {"timepoint": "day4p5", "well": "B02", "organoid_id": "124"},
    {"timepoint": "day4p5", "well": "B04", "organoid_id": "4"},
    {"timepoint": "day4p5", "well": "B05", "organoid_id": "54"},
    {"timepoint": "day4p5", "well": "B02", "organoid_id": "115"},
    {"timepoint": "day4p5", "well": "B02", "organoid_id": "100"},
    {"timepoint": "day4p5", "well": "B02", "organoid_id": "31"},
]

# Optional escape hatch for unusual paths: {"well/organoid_id": "/full/path/file.vtp"}
MESH_PATH_OVERRIDES = {}

print("PROJECT_ROOT     =", PROJECT_ROOT)
print("MESH_DATA_DIR   =", MESH_DATA_DIR)
print("MESH_CONFIG_PATH=", MESH_CONFIG_PATH)
print("VOCAB_PATH      =", VOCAB_PATH)
print("n_organoids     =", len(ORGANOID_SPECS))


PROJECT_ROOT     = /home/fmoller/Projects/LearningOrganoids/OrganoGraph
MESH_DATA_DIR   = /home/fmoller/Projects/LearningOrganoids/NicoleData/20251201/fractal_output
MESH_CONFIG_PATH= /home/fmoller/Projects/LearningOrganoids/NicoleData/20251201/mesh_config.json
VOCAB_PATH      = /home/fmoller/Projects/LearningOrganoids/OrganoGraph/sim/vocab_with_meta.npz
n_organoids     = 7


In [3]:
# --- Build mesh paths from dataset/timepoint/well/organoid ids ---
mesh_cfg = load_mesh_dataset_config(str(MESH_CONFIG_PATH))


def mesh_path_from_spec(spec):
    timepoint = str(spec["timepoint"])
    well = str(spec["well"])
    organoid_id = str(spec["organoid_id"])
    override_key = f"{well}/{organoid_id}"
    if override_key in MESH_PATH_OVERRIDES:
        return Path(MESH_PATH_OVERRIDES[override_key])

    if timepoint not in mesh_cfg["zarr_name_by_tp"]:
        raise KeyError(f"timepoint={timepoint!r} is not present in {MESH_CONFIG_PATH}")
    if len(well) < 2:
        raise ValueError(f"well should look like 'B03', got {well!r}")

    return (
        Path(MESH_DATA_DIR)
        / timepoint
        / mesh_cfg["zarr_name_by_tp"][timepoint]
        / well[0]
        / well[1:]
        / mesh_cfg["round_by_tp"][timepoint]
        / "meshes"
        / mesh_cfg["meshname_by_tp"][timepoint]
        / f"{organoid_id}.vtp"
    )


organoid_records = []
missing = []
for spec in ORGANOID_SPECS:
    mesh_path = mesh_path_from_spec(spec)
    rec = dict(spec)
    rec["mesh_path"] = str(mesh_path)
    rec["label_uid"] = f"{rec['timepoint']}_{rec['well']}_{rec['organoid_id']}"
    if mesh_path.exists():
        try:
            parsed = parse_mesh_path(str(mesh_path))
            rec["label_uid"] = parsed.get("label_uid", rec["label_uid"])
        except Exception:
            pass
        organoid_records.append(rec)
    else:
        missing.append(rec)

if missing:
    print("Missing requested mesh paths:")
    for rec in missing:
        print(" ", rec["mesh_path"])
        nearby = discover_mesh_paths(
            data_dir=str(MESH_DATA_DIR),
            timepoints=[rec["timepoint"]],
            zarr_names=mesh_cfg["zarr_name_by_tp"],
            rounds=mesh_cfg["round_by_tp"],
            meshes=mesh_cfg["meshname_by_tp"],
            wells={rec["timepoint"]: [rec["well"]]},
        )
        print(f"   available IDs in {rec['timepoint']}/{rec['well']}:", [Path(p).stem for p in nearby[:20]])
    raise FileNotFoundError("One or more requested organoids were not found.")



## 2) Load and prepare meshes

The HKS-based crypt detector needs a Laplace-Beltrami eigendecomposition for each mesh. Meshes are loaded lazily and cached as the batch loop runs. If smoothing is enabled here, the low-pass reconstructed mesh is used for detection, skeleton construction, and plotting.


In [4]:
NORMALIZE_MESH = True
NORMALIZE_SCALE = 10.0
EIGEN_K = 225

# --- Optional mesh smoothing before any detection/skeleton/plotting step ---
# If enabled, the mesh vertices are replaced by a low-pass spectral reconstruction.
SMOOTH_MESH = True
SMOOTH_LMAX = 12
SMOOTH_EIGEN_K = None  # None uses EIGEN_K after smoothing.

vocab = np.load(str(VOCAB_PATH), allow_pickle=True)
MESH_CACHE = {}


def _clamped_eigen_k(mesh, requested_k):
    n_vertices = int(np.asarray(mesh.v).shape[0])
    return max(2, min(int(requested_k), n_vertices - 2))


def _reset_spectral_state(mesh):
    mesh.laplacian = None
    mesh.mass_matrix = None
    mesh.eigvals = None
    mesh.eigvecs = None
    mesh.coeffs_v = None
    mesh.lmax = None


def _ensure_mesh_eigendecomposition(mesh, requested_k):
    k = _clamped_eigen_k(mesh, requested_k)
    if mesh.eigvals is None or mesh.eigvecs is None or mesh.eigvecs.shape[1] < k:
        _reset_spectral_state(mesh)
        mesh._eig_decomp(k=k)
    return k


def _smooth_mesh_low_pass(mesh):
    lmax = int(SMOOTH_LMAX)
    if lmax < 1:
        raise ValueError("SMOOTH_LMAX must be at least 1")

    coeff_k = int(lmax ** 2)
    _ensure_mesh_eigendecomposition(mesh, max(EIGEN_K, coeff_k))
    mesh.compute_spectral_coefficients(lmax=lmax)
    mesh.v = np.asarray(mesh.reconstruct_from_coeffs(mesh.coeffs_v, lmax=lmax), dtype=float)

    # The spectral operators must match the smoothed coordinates used downstream.
    _reset_spectral_state(mesh)
    _ensure_mesh_eigendecomposition(mesh, SMOOTH_EIGEN_K or EIGEN_K)
    return mesh


def load_prepared_mesh(record):
    label_uid = record["label_uid"]
    if label_uid in MESH_CACHE:
        return MESH_CACHE[label_uid]

    mesh = OrganoidMesh(str(record["mesh_path"]))
    mesh.label_uid = label_uid

    if NORMALIZE_MESH:
        mesh.normalize_inplace(scale=NORMALIZE_SCALE, center="mean")

    if SMOOTH_MESH:
        _smooth_mesh_low_pass(mesh)
    else:
        # This also builds the mass matrix used by vertex_areas().
        _ensure_mesh_eigendecomposition(mesh, EIGEN_K)
    MESH_CACHE[label_uid] = mesh
    return mesh


print("vocab entries:", np.asarray(vocab["vocab"]).shape)
print("batch size   :", len(organoid_records))


vocab entries: (8, 20)
batch size   : 7


## 3) Tuning parameters

The skeleton adapter intentionally reruns detection from the fresh HKS candidate screen. This is useful for split crypts: the initial candidate patch can serve as the skeleton trunk/stem, while refinement identifies daughter tips.

Mesh smoothing is controlled in the preparation cell above so the same vertices are used everywhere. `DETECTION_KWARGS` controls HKS candidate detection, boundary-distance neckline normalization, post-neck HKS-guided skeleton-tip refinement with an optional minimum HKS-increase threshold, and split-stem validation by parent-patch boundary growth, including optional pre-growth perimeter smoothing and an absolute mesh-fraction growth cap. `FILTER_KWARGS` controls reusable crypt filters. `BUILD_KWARGS` controls graph construction, especially optional bend-node insertion and region-centroid refinement of body/branch nodes.

`bend_strategy="none"` builds direct neck-to-tip edges. Use `bend_strategy="crypt_centroid"` to insert a `crypt` node at the centroid of the detected crypt vertices, or `midpoint` / `crypt_centroid_midsection` to insert a `bend` node.


In [5]:
# --- Crypt detection parameters ---
DETECTION_KWARGS = dict(
    L_ref=None,
    crypt_vocab_idx=None,
    threshold=0.5,
    refine_crypts=True,
    refine_threshold=0.00,
    refine_only_if_area_at_least=5.0,
    min_refined_frac_of_parent=0.05,
    geodesic_kwargs=None,
    final_tip_hks_time=1.0,
    final_tip_bottom_fraction=0.8,
    final_tip_min_hks_percent_increase=5.0,  # require updated-tip HKS to exceed initial-tip HKS by this percent
    extend_max=2.0,
    disc_resolution=200,
    neck_search_interval=(0.8, 2.0),
    # Branch refinement: keep split hierarchy only if parent-patch growth
    # finds a robust global boundary-length minimum.
    validate_split_stems=True,
    split_growth_max_size_factor=3.0,
    split_growth_max_mesh_fraction=0.40,
    split_growth_smooth_perimeter=True,
    split_growth_smoothing_tolerance=0.0,
    split_growth_min_decrease_fraction=0.0,
    split_growth_min_prominence_fraction=0.01,
    split_growth_robust_window=1,
)

# --- Candidate filters ---
FILTER_KWARGS = dict(
    use_hks_filter=True,
    min_percent_greater=2.0,
    hks_t_min=None,
    hks_t_max=10.0,
    use_size_filter=True,
    min_patch_verts=25,
    min_patch_area=5.0,
)

# --- Skeleton graph parameters ---
BUILD_KWARGS = dict(
    body_center=None,
    # "none" gives direct neck-to-tip edges; "crypt_centroid" inserts the central crypt node.
    bend_strategy="crypt_centroid",  # "none", "crypt_centroid", "midpoint", "crypt_centroid_midsection"
    refine_body_center_from_necks=True,
    refine_branch_centers_from_necks=True,
)

def make_filter_list(**kw):
    filters = []
    if kw.get("use_hks_filter", True):
        filters.append(
            lambda patches, **inner: filter_crypts_by_hks_percent(
                patches,
                min_percent_greater=kw["min_percent_greater"],
                t_min=kw.get("hks_t_min"),
                t_max=kw.get("hks_t_max"),
                **inner,
            )
        )
    if kw.get("use_size_filter", True):
        filters.append(
            lambda patches, **inner: filter_crypts_by_size(
                patches,
                min_patch_verts=kw["min_patch_verts"],
                min_patch_area=kw.get("min_patch_area"),
                **inner,
            )
        )
    return filters or None

## 4) Build skeletons for the batch

The function below is the main tuning loop. It runs detection and skeleton construction for every organoid in `organoid_records`, then displays each skeleton figure.

In [ ]:
def run_skeleton_pipeline(
    record,
    *,
    detection_kwargs=None,
    filter_kwargs=None,
    build_kwargs=None,
    show_plots=True,
):
    detection_kwargs = dict(DETECTION_KWARGS if detection_kwargs is None else detection_kwargs)
    filter_kwargs = dict(FILTER_KWARGS if filter_kwargs is None else filter_kwargs)
    build_kwargs = dict(BUILD_KWARGS if build_kwargs is None else build_kwargs)
    mesh = load_prepared_mesh(record)

    filter_fn_list = make_filter_list(**filter_kwargs)

    detections, skel_vars = detect_crypts_for_skeleton(
        mesh,
        vocab,
        geodesic_fn=compute_geodesics_dijkstra,
        filter_fn_list=filter_fn_list,
        return_intermediates=True,
        **detection_kwargs,
    )

    build_kwargs["metadata"] = {
        "label_uid": record["label_uid"],
        "mesh_path": record["mesh_path"],
        "timepoint": record["timepoint"],
        "well": record["well"],
        "organoid_id": record["organoid_id"],
    }
    graph = build_skeleton_from_crypt_detections(
        vertices=mesh.v,
        faces=mesh.f,
        crypt_detections=detections,
        **build_kwargs,
    )

    if show_plots:
        display(
            plot_mesh_with_skeleton(
                mesh.v,
                mesh.f,
                graph,
                backend="plotly",
                mesh_alpha=0.22,
                show_node_labels=True,
            )
        )

    return dict(
        record=record,
        mesh=mesh,
        detections=detections,
        graph=graph,
        skel_vars=skel_vars,
    )


batch_results = []
for record in organoid_records:
    display(HTML(f"<h3>{record['label_uid']}</h3>"))
    result = run_skeleton_pipeline(record, show_plots=True)
    batch_results.append(result)


## Notes and TODOs

- `SMOOTH_MESH` is applied during mesh preparation, before HKS detection; when enabled, the smoothed vertices are also used for skeleton building and plotting.
- Neck nodes are placed at the centroid of the estimated neckline ring when a normalized distance field is available, so they should lie inside the mesh rather than on the surface.
- Intermediate nodes are optional. `crypt_centroid` inserts a central crypt node; `midpoint` and `crypt_centroid_midsection` insert bend nodes.
- Branch refinement grows parent candidate patches and keeps split topology only when the boundary-length trace has a robust internal global minimum.
- Branch points for split crypts use explicit split coordinates if provided; otherwise the builder falls back to a stem centroid or a midpoint between neck and daughter tips.
- Future primitive fitting can attach to `node.primitive_attachment` or `edge.primitive_attachment` without changing the skeleton topology.

## Adding primitive attachments to skeletons

This section fits simple geometric primitives to an already-built skeleton. The component extraction below uses neck-bounded regions from the skeleton detections. Body primitives are fit after cutting away every root crypt or branch at its body-side neckline; branch primitives are fit after cutting daughter crypts away at their daughter necklines.

The first primitive layer uses PCA ellipsoids for body/branch blobs and tapered capped tubes for crypt paths. Tube parameters are fitted degrees of freedom; quantities such as length, bend angle, tortuosity, constriction ratio, and taper ratio are derived afterward.


In [ ]:
def attach_primitives_to_result(result, *, radius_quantile=0.5):
    mesh = result["mesh"]
    graph = result["graph"]
    components = primitive_components_from_crypt_detections(
        mesh.v,
        result["detections"],
        graph=graph,
    )

    attach_body_primitive(graph, mesh.v, components["body"])
    if components["branches"]:
        attach_branch_primitives(graph, mesh.v, components["branches"])
    if components["crypts"]:
        attach_crypt_tube_primitives(
            graph,
            mesh.v,
            components["crypts"],
            radius_quantile=radius_quantile,
        )

    return components


In [ ]:
# Fit and plot primitives for every already-built skeleton in the batch.
primitive_results = []
for primitive_result in batch_results:
    record = primitive_result["record"]
    attach_primitives_to_result(primitive_result, radius_quantile=0.5)
    primitive_results.append(primitive_result)

    display(HTML(f"<h3>{record['label_uid']} primitives</h3>"))
    display(
        plot_mesh_with_skeleton_and_primitives(
            primitive_result["mesh"].v,
            primitive_result["mesh"].f,
            primitive_result["graph"],
            backend="plotly",
            mesh_alpha=0.12,
            primitive_alpha=0.32,
            show_node_labels=True,
        )
    )
